# Обучение YOLOv4-tiny (Darknet) на своём датасете

Датасет — `dataset.zip` из Autolabel Studio (вкладка «6. Обучение» → «Собрать dataset.zip»).

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**.

Порядок: запускайте ячейки сверху вниз (Shift+Enter). Веса сохраняются на Google Диск,
поэтому если Colab отключился — выполните шаги 1–5 заново и запустите **шаг 7 «Продолжить»**.

## 0. Настройки

In [ ]:
import os, re, glob, shutil, random

# Папка на Google Диске, куда сохраняются веса (переживёт отключение Colab)
BACKUP = '/content/drive/MyDrive/yolo_backup'

# Где лежит датасет. Если файла нет — на шаге 5 появится кнопка загрузки с компьютера.
# Удобнее один раз положить dataset.zip на Google Диск и указать путь:
DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'

CFG = 'cfg/yolov4-tiny-custom.cfg'
DATA = 'data/obj.data'
best = f'{BACKUP}/yolov4-tiny-custom_best.weights'
last = f'{BACKUP}/yolov4-tiny-custom_last.weights'

## 1. Проверить видеокарту

In [ ]:
!nvidia-smi

## 2. Скачать и собрать Darknet (≈3–5 минут)

In [ ]:
import os, re, subprocess
os.chdir('/content')
if not os.path.exists('/content/darknet'):
    !git clone -q https://github.com/AlexeyAB/darknet
%cd /content/darknet

# включаем GPU, cuDNN, OpenCV и собираем под видеокарту этой машины
cc = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']).decode().split()[0].replace('.', '')
mk = open('Makefile').read()
for k in ('GPU', 'CUDNN', 'CUDNN_HALF', 'OPENCV'):
    mk = re.sub(rf'^{k}=0', f'{k}=1', mk, flags=re.M)
mk = re.sub(r'^ARCH=.*?(?<!\\)\n', f'ARCH= -gencode arch=compute_{cc},code=[sm_{cc},compute_{cc}]\n', mk, count=1, flags=re.S | re.M)
# CUDNN_HALF дописывает compute_70 — новая CUDA в Colab его уже не знает («Unsupported gpu architecture»)
mk = re.sub(r'^[ \t]*ARCH\+=.*\n', '', mk, flags=re.M)
open('Makefile', 'w').write(mk)
print('Видеокарта: compute', cc)

!apt-get -qq install -y libopencv-dev > /dev/null 2>&1
!nvcc --version | tail -2
!make clean > /dev/null 2>&1; make -j$(nproc) > build.log 2>&1 || grep -E -m 20 "error|fatal|Error" build.log
print('✅ Darknet собран' if os.path.exists('darknet') else '❌ Сборка не удалась — смотрите ошибки выше (build.log)')

## 3. Стартовые веса yolov4-tiny

In [ ]:
%cd /content/darknet
if not os.path.exists('yolov4-tiny.conv.29'):
    !wget -q https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29
print('✅ готово' if os.path.exists('yolov4-tiny.conv.29') else '❌ не скачалось')

## 4. Подключить Google Диск

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(BACKUP, exist_ok=True)
print('Веса будут сохраняться в', BACKUP)

## 5. Загрузить и распаковать датасет

In [ ]:
%cd /content/darknet
import shutil, glob
zip_path = DATASET_ZIP
if not os.path.exists(zip_path):
    from google.colab import files
    print('dataset.zip не найден на Диске — выберите файл на компьютере:')
    up = files.upload()
    zip_path = '/content/darknet/' + list(up)[0]

shutil.rmtree('data/obj', ignore_errors=True)
!unzip -q -o "{zip_path}" -d data/
!cp data/yolov4-tiny-custom.cfg {CFG}

# веса — на Google Диск
txt = open(DATA).read()
txt = re.sub(r'backup\s*=.*', f'backup = {BACKUP}/', txt)
open(DATA, 'w').write(txt)

# проверка датасета
names = [l.strip() for l in open('data/obj.names') if l.strip()]
imgs = glob.glob('data/obj/*.jpg') + glob.glob('data/obj/*.png')
train = [l for l in open('data/train.txt') if l.strip()]
valid = [l for l in open('data/valid.txt') if l.strip()]
missing = [l.strip() for l in train + valid if not os.path.exists(l.strip())]
cnt = {}
for t in glob.glob('data/obj/*.txt'):
    for l in open(t):
        if l.strip():
            c = int(l.split()[0]); cnt[c] = cnt.get(c, 0) + 1
cfg = open(CFG).read()
print(open(DATA).read())
print(f'Классов: {len(names)}, картинок: {len(imgs)}, train: {len(train)}, valid: {len(valid)}')
for i, n in enumerate(names):
    print(f'  {i}: {n:<28} рамок: {cnt.get(i, 0)}')
print('classes в cfg:', re.findall(r'classes\s*=\s*(\d+)', cfg), ' filters перед [yolo]: должно быть', (len(names) + 5) * 3)
print('max_batches:', re.findall(r'max_batches\s*=\s*(\d+)', cfg))
print('❌ Битых путей в train/valid: %d' % len(missing) if missing else '✅ Все пути в train/valid на месте')

## 6. Обучение с нуля

Идёт 1–3 часа. Каждые ~1000 итераций сохраняются `_last`, `_best` (по mAP) и `_1000`, `_2000`… в `BACKUP`.
В выводе оставлены только строки с loss и mAP, полный лог — `train.log`.
Хороший признак: `avg loss` падает ниже ~1, `mAP@0.50` растёт к 0.8–0.95.

In [ ]:
%cd /content/darknet
!stdbuf -oL ./darknet detector train {DATA} {CFG} yolov4-tiny.conv.29 -dont_show -map 2>&1 | tee train.log | grep --line-buffered -E "avg loss|mAP@0.50|mean_average_precision|Saving weights|rror|Can't|Couldn't"

## 7. Продолжить после отключения Colab

Сначала заново выполните шаги 0–5, потом эту ячейку.

In [ ]:
%cd /content/darknet
if os.path.exists(last):
    !stdbuf -oL ./darknet detector train {DATA} {CFG} "{last}" -dont_show -map 2>&1 | tee -a train.log | grep --line-buffered -E "avg loss|mAP@0.50|mean_average_precision|Saving weights|rror|Can't|Couldn't"
else:
    print('Нет', last, '— запустите шаг 6')

## 8. График обучения

In [ ]:
%cd /content/darknet
from IPython.display import Image, display
charts = sorted(glob.glob('chart*.png'), key=os.path.getmtime)
if charts:
    shutil.copy(charts[-1], f'{BACKUP}/chart.png')
    display(Image(charts[-1]))
else:
    print('График появится после начала обучения')

## 9. Точность (mAP) лучших весов

In [ ]:
%cd /content/darknet
!./darknet detector map {DATA} {CFG} "{best}" -dont_show 2>&1 | grep -E "class_id|mAP@|precision|recall|F1"

## 10. Проверка на случайной картинке из valid

In [ ]:
%cd /content/darknet
img = random.choice([l.strip() for l in open('data/valid.txt') if l.strip()])
!./darknet detector test {DATA} {CFG} "{best}" "{img}" -dont_show -thresh 0.4 -ext_output 2>&1 | tail -n 12
display(Image('predictions.jpg'))

## 11. Проверка на своём видео

Выберите видео — результат с рамками скачается как `result.mp4`.

In [ ]:
%cd /content/darknet
from google.colab import files
up = files.upload()
video = list(up)[0]
!./darknet detector demo {DATA} {CFG} "{best}" "{video}" -dont_show -thresh 0.4 -out_filename result.avi > demo.log 2>&1
!ffmpeg -y -loglevel error -i result.avi -vcodec libx264 -crf 26 result.mp4
files.download('result.mp4')

## 12. Забрать готовую модель

In [ ]:
%cd /content/darknet
out = f'{BACKUP}/model'
os.makedirs(out, exist_ok=True)
for f in (best, CFG, 'data/obj.names', 'data/obj.data'):
    shutil.copy(f, out)
# для запуска на своём компьютере: batch=1, subdivisions=1
c = open(f'{out}/yolov4-tiny-custom.cfg').read()
c = re.sub(r'^batch=\d+', 'batch=1', c, flags=re.M)
c = re.sub(r'^subdivisions=\d+', 'subdivisions=1', c, flags=re.M)
open(f'{out}/yolov4-tiny-custom.cfg', 'w').write(c)
!cd "{BACKUP}" && zip -q -r model.zip model
print('Модель на Диске:', out)
from google.colab import files
files.download(f'{BACKUP}/model.zip')